# CNN Basil Leaf Health Classifier

Pipeline responsibility: Deep Learning Image Classification  
Model assignment: Convolutional Neural Network (`BasilLeafCNN`)

This notebook trains a CNN on the labelled basil leaf images under `data/raw/` (`Healthy` vs `Unhealthy`).

It reuses the shared audit / grouped holdout split from `parts/_pipeline.py` so results are comparable with the six classical models in `parts/`.

Artefacts are written to three folders:

- `CNN/outputs/` — trained weights (`cnn_model.pt`)
- `CNN/analytics/` — training curves, confusion matrix, history, dataset roles
- `CNN/results/` — holdout metrics, per-class scores, predictions, model comparison


## 1. Setup

Requires PyTorch in the project `.venv`:

```powershell
pip install torch --index-url https://download.pytorch.org/whl/cpu
```


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / "data" / "raw").is_dir() and (p / "Basil_Leaf_ML_Workflow.ipynb").exists()),
    None,
)
if ROOT is None:
    ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root containing data/raw.")

CNN_DIR = ROOT / "CNN"
OUT = CNN_DIR / "outputs"
ANALYTICS = CNN_DIR / "analytics"
RESULTS = CNN_DIR / "results"
for folder in (OUT, ANALYTICS, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CNN_DIR))
sys.path.insert(0, str(ROOT / "parts"))

print("Project root:", ROOT)
print("Outputs:", OUT)
print("Analytics:", ANALYTICS)
print("Results:", RESULTS)
print("Python:", sys.executable)

import torch
print("torch:", torch.__version__)


## 2. Dataset audit and split

The CNN uses the same four labelled folders and grouped holdout as the classical notebooks. Images are resized to 128x128 RGB inside `LeafDataset` (no handcrafted RGB/HSV/LBP/HOG features).


In [ ]:
from _pipeline import CLASSES, SEED, audit_dataset, make_splits
import numpy as np

data_dir = ROOT / "data" / "raw"
frame, audit = audit_dataset(data_dir)
dev, test, cv = make_splits(frame)
train_rel, val_rel = cv[0]
train_idx = np.asarray(dev)[train_rel]
val_idx = np.asarray(dev)[val_rel]
test_idx = np.asarray(test)

print("Unique images:", audit["valid_unique_count"])
print("Class counts:", audit["class_counts"])
print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")
print("Classes:", CLASSES, "| seed:", SEED)
display(frame.groupby("label").size().rename("count").to_frame())


## 3. CNN architecture

Custom 4-block CNN:

1. Conv 3→32 + BatchNorm + ReLU + MaxPool
2. Conv 32→64 + BatchNorm + ReLU + MaxPool
3. Conv 64→128 + BatchNorm + ReLU + MaxPool
4. Conv 128→256 + BatchNorm + ReLU + MaxPool (8x8 feature map)
5. Dropout 0.4 → Linear 16384→256 → ReLU → Dropout 0.3 → Linear 256→2

Training: 20 epochs, Adam (lr=1e-3, weight decay=1e-4), class-weighted CrossEntropyLoss, ReduceLROnPlateau on validation macro-F1. The best validation checkpoint is evaluated on the holdout set.


In [ ]:
from train_cnn import BasilLeafCNN, IMG_SIZE, BATCH_SIZE, EPOCHS, LR, MODEL_NAME

model = BasilLeafCNN(n_classes=len(CLASSES))
n_params = sum(p.numel() for p in model.parameters())
print(MODEL_NAME)
print("Image size:", IMG_SIZE, "| batch:", BATCH_SIZE, "| epochs:", EPOCHS, "| lr:", LR)
print("Trainable parameters:", n_params)
print(model)


## 4. Train CNN (or load existing run)

`train_cnn.main()` writes the model, analytics plots, and results. If a checkpoint already exists this cell skips retraining (~11 minutes on CPU). Delete `CNN/outputs/cnn_model.pt` to force a new run.


In [ ]:
from train_cnn import main as train_cnn_main

ckpt = OUT / "cnn_model.pt"
metrics_path = RESULTS / "cnn_metrics.json"
if ckpt.exists() and metrics_path.exists():
    print("Found existing trained model — skipping retrain.")
    print("Checkpoint:", ckpt)
    print("Metrics:", metrics_path)
else:
    train_cnn_main()
    print("Training finished.")


## 5. Analytics

Training history, loss/accuracy curves, and the holdout confusion matrix live under `CNN/analytics/`.


In [ ]:
history = pd.read_csv(ANALYTICS / "training_history.csv")
display(Markdown("### Per-epoch history"))
display(history)
best = history.loc[history["val_macro_f1"].idxmax()]
print(f"Best val macro-F1: {best.val_macro_f1:.4f} at epoch {int(best.epoch)} (lr={best.lr})")

display(Markdown("### Training curves"))
display(Image(filename=str(ANALYTICS / "training_curves.png")))
display(Markdown("### Confusion matrix"))
display(Image(filename=str(ANALYTICS / "confusion_matrix.png")))


## 6. Results

Holdout metrics, per-class scores, and comparison against the six classical models live under `CNN/results/`.


In [ ]:
metrics = json.loads((RESULTS / "cnn_metrics.json").read_text(encoding="utf-8"))
print("Model:", metrics["model_name"])
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}")
print(f"95% CI: {metrics['bootstrap_95_ci']}")
print(f"Fit time (s): {metrics['fit_time_seconds']:.1f}")
print("Confusion matrix:", metrics["confusion_matrix"])
print(f"n_train={metrics['n_train']} n_val={metrics['n_val']} n_test={metrics['n_test']}")

display(Markdown("### Per-class performance"))
display(pd.DataFrame(json.loads((RESULTS / "class_performance.json").read_text(encoding="utf-8"))).T)

display(Markdown("### Model comparison (classical + CNN)"))
display(pd.read_csv(RESULTS / "model_comparison_with_cnn.csv"))

preds = pd.read_csv(RESULTS / "test_predictions.csv")
print("Holdout predictions:", len(preds), "| errors:", int((~preds.correct).sum()))
display(preds.loc[~preds.correct, ["path", "y_true", "y_pred"]].reset_index(drop=True))


## 7. Interpretation

On the same grouped holdout (225 images), **BasilLeafCNN** reaches **98.67% accuracy** and **0.9864 macro F1**, slightly above Random Forest / Gradient Boosting (98.22% / 0.9817–0.9818).

The network learns spatial leaf patterns directly from pixels instead of handcrafted RGB/HSV/LBP/HOG features. Cost is training time (~11 minutes on CPU vs seconds for the tree models).

Open `analytics/ANALYTICS.md` and `results/RESULTS.md` for the written summary.
